In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/adsb_data/silver/flights")

In [0]:
from datetime import datetime
mock_flights = [
    ("FLT001", "a1b2c3","RAM201", 1775058950, 1775069750, 9000.0, 11000.0, 250.0, 280.0,"Morocco", 31.6069, -8.0363, 50.0379, 8.5622, 120, 180.0, "GMMX", "EDDF", 2800.0),
    ("FLT002", "3f8a12","RAM202", 1775062550, 1775073050, 8800.0, 10800.0, 245.0, 275.0, "Morocco",31.6069, -8.0363, 50.0379, 8.5622, 118, 175.0, "GMMX", "EDDF", 2780.0),
    ("FLT003", "4ca2b1","DLH442", 1775072950, 1775083550, 9100.0, 11100.0, 255.0, 285.0, "Germany",50.0379,  8.5622, 31.6069, -8.0363, 122, 185.0, "EDDF", "GMMX", 2820.0),
    ("FLT004", "4d3a1b","DLH443", 1775055350, 1775065550, 8700.0, 10700.0, 240.0, 270.0, "Germany",50.0379,  8.5622, 31.6069, -8.0363, 115, 170.0, "EDDF", "GMMX", 2760.0),

    ("FLT005", "3a1b2c","IBE334", 1775029600, 1775036800, 9200.0, 11200.0, 248.0, 278.0, "Spain",40.4719, -3.5626, 49.0097, 2.5478, 110, 120.0, "LEMD", "LFPG", 1050.0),
    ("FLT006", "4f2a1b","AFR772", 1775036800, 1775041200, 8500.0, 10500.0, 230.0, 260.0, "France",49.0097,  2.5478, 50.9010, 4.4844, 90,   73.0,  "LFPG", "EBBR", 260.0),
    ("FLT007", "3c1a2b","BEL231", 1775041200, 1775048400, 8800.0, 10800.0, 240.0, 270.0, "Belgium",50.9010,  4.4844, 48.3538, 11.786, 105,  120.0, "EBBR", "EDDM", 650.0),
    ("FLT008", "a2b3c4","DLH444", 1775048400, 1775054400, 9000.0, 11000.0, 245.0, 275.0, "Germany",48.3538, 11.7861, 43.6293, 1.3677, 100,  100.0, "EDDM", "LFBO", 580.0),
    ("FLT009", "4b1a3c","AFR991", 1775054400, 1775061600, 9100.0, 11100.0, 250.0, 280.0, "France",43.6293,  1.3677, 40.4719, -3.5626,112,  120.0, "LFBO", "LEMD", 1000.0),
    ("FLT010", "3d2a1b","AFR773", 1775033000, 1775040200, 9300.0, 11300.0, 252.0, 282.0, "France",49.0097,  2.5478, 48.3538, 11.7861, 108, 120.0, "LFPG", "EDDM", 680.0),
    ("FLT011", "a3c1b2","IBE335", 1775040000, 1775045600, 8600.0, 10600.0, 235.0, 265.0, "Spain",40.4719, -3.5626, 50.9010, 4.4844, 95,   93.0, "LEMD", "EBBR", 1500.0),
]

mock_flights_df = spark.createDataFrame(mock_flights, schema=df.schema)
df = df.union(mock_flights_df)

In [0]:
from pyspark.sql.functions import to_date as to_date_spark
from pyspark.sql.functions import from_unixtime, col
df = df.withColumn("flight_date", to_date_spark(from_unixtime(col("departure_ts"))))

# Airports stats

In [0]:
from pyspark.sql.functions import count, countDistinct
departures_df = df.filter(col("origin_airport").isNotNull()).groupBy("flight_date", "origin_airport") \
                  .agg(count("*").alias("num_departures"),
                       countDistinct("icao24").alias("departing_aircraft"))

In [0]:
arrivals_df = df.filter(col("dest_airport").isNotNull()).groupBy("flight_date", "dest_airport") \
                  .agg(count("*").alias("num_arrivals"),
                       countDistinct("icao24").alias("arriving_aircraft"))

In [0]:
from pyspark.sql.functions import coalesce

airport_stats_df = departures_df.alias("dep").join(
    arrivals_df.alias("arr"),
    (col("dep.flight_date") == col("arr.flight_date")) &
    (col("dep.origin_airport") == col("arr.dest_airport")),
    "outer"
) \
.fillna(0, subset=["num_departures", "num_arrivals"]) \
.withColumn("total_movements", col("num_departures") + col("num_arrivals")) \
.select(
    coalesce(col("dep.flight_date"), col("arr.flight_date")).alias("flight_date"),
    coalesce(col("dep.origin_airport"), col("arr.dest_airport")).alias("airport_icao"),
    "num_departures",
    "num_arrivals",
    "total_movements",
    "arriving_aircraft",
    "departing_aircraft"
)

In [0]:
from pyspark.sql.functions import avg
# Flights where airport appears as origin
origin_flights = df.withColumn("airport_icao", col("origin_airport"))

# Flights where airport appears as destination
dest_flights = df.withColumn("airport_icao", col("dest_airport"))


avg_stats_df = origin_flights.union(dest_flights) \
                .groupBy("flight_date", "airport_icao") \
                .agg(
                        avg("duration_min").alias("avg_duration"),
                        avg(col("traveled_distance")).alias("avg_distance"),
                        avg("avg_altitude").alias("avg_cruise_altitude"),
                        avg("avg_speed").alias("avg_cruise_speed")
                )

unique_aircraft_df = origin_flights.select("flight_date", "airport_icao", "icao24") \
                    .union(dest_flights.select("flight_date", "airport_icao", "icao24")) \
                    .groupBy("flight_date", "airport_icao") \
                    .agg(countDistinct("icao24").alias("unique_aircraft"))


daily_airport_stats = airport_stats_df \
    .join(avg_stats_df, ["flight_date", "airport_icao"], "left") \
    .join(unique_aircraft_df, ["flight_date", "airport_icao"], "left")

daily_airport_stats = daily_airport_stats.select(
    "flight_date",
    "airport_icao",
    "num_departures",
    "num_arrivals",
    "total_movements",
    "avg_duration",
    "avg_distance",
    "avg_cruise_altitude",
    "avg_cruise_speed",
    "arriving_aircraft",
    "departing_aircraft",
    "unique_aircraft",
)

# Carrier performance

In [0]:
from pyspark.sql.functions import substring

CARRIER_NUMB_FLIGHTS_THRESHOLD = 3
# (avg_speed * avg_distance) / (avg_duration * avg_altitude) this is a temp formula change it with a forumla tht reflects reality
carrier_performance_df = df.filter(col("callsign").isNotNull())\
     .withColumn("carrier_code", substring("callsign", 1, 3))\
     .groupBy("flight_date", "carrier_code","origin_country") \
     .agg(count("*").alias("num_flights"),
          avg("duration_min").alias("avg_duration"),
          avg("traveled_distance").alias("avg_distance"),
          avg("avg_altitude").alias("avg_altitude"),
          avg("avg_speed").alias("avg_speed")
          )\
     .withColumn("efficiency_score", (col("avg_speed")*col("avg_distance")) / ( col("avg_duration")*col("avg_altitude") ) )\
     .filter(col("num_flights") >= CARRIER_NUMB_FLIGHTS_THRESHOLD)

# Airspace congestion

In [0]:
BRONZE_PATH = "/Volumes/workspace/default/adsb_data/bronze/live_states"
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

In [0]:
from pyspark.sql.functions import floor, when
CONGESTION_LEVEL_HIGH_THRESHOLD = 50
CONGESTION_LEVEL_MEDIUM_THRESHOLD = 20 
airspace_congestion_df = bronze_df.filter(col("on_ground") == False)\
                                  .withColumn("hour_date", to_date_spark(from_unixtime(col("api_timestamp"))))\
                                  .withColumn("grid_lat", floor(col("latitude")))\
                                  .withColumn("grid_lon", floor(col("longitude")))\
                                  .groupBy("hour_date", "ingest_hour", "grid_lat", "grid_lon")\
                                  .agg(
                                      countDistinct("icao24").alias("aircraft_count"),
                                      avg("baro_altitude").alias("avg_altitude"),
                                      avg("velocity").alias("avg_speed"),
                                  ).withColumn("congestion_level",
                                               when( col("aircraft_count") > CONGESTION_LEVEL_HIGH_THRESHOLD, "HIGH")
                                               .when( col("aircraft_count") > CONGESTION_LEVEL_MEDIUM_THRESHOLD,"MEDIUM"
                                               ).otherwise("LOW")
                                    )

# Writing tables to gold

In [0]:
from delta.tables import DeltaTable

GOLD_PATH = "/Volumes/workspace/default/adsb_data/gold"

# daily_airport_stats
daily_airport_stats.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/daily_airport_stats")

# carrier_performance
carrier_performance_df.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/carrier_performance")

# airspace_congestion - MERGE
if DeltaTable.isDeltaTable(spark, f"{GOLD_PATH}/airspace_congestion"):
    DeltaTable.forPath(spark, f"{GOLD_PATH}/airspace_congestion").alias("target")\
        .merge(
            airspace_congestion_df.alias("source"),
            "target.hour_date = source.hour_date AND target.ingest_hour = source.ingest_hour AND target.grid_lat = source.grid_lat AND target.grid_lon = source.grid_lon"
        )\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    airspace_congestion_df.write.format("delta").save(f"{GOLD_PATH}/airspace_congestion")